# 🏪 Kasir Warung — Notebook Per-Bab

Notebook ini adalah versi Jupyter dari `kasir_warung.py`, dipecah per bagian/bab sesuai struktur kode aslinya.

**Cara pakai:** jalankan seluruh cell secara berurutan (Run All), lalu panggil `main()` di cell terakhir untuk menjalankan program secara interaktif.

In [ ]:
"""
kasir_warung.py
===============
Aplikasi Kasir Warung Sederhana — Tugas Akhir Algoritma & Pemrograman
Universitas Al Azhar Indonesia — Semester Genap 2025/2026

Mahasiswa : Randy Ramadhan, Wulan Purnamasari, Bily Nur Sepdiansyah, Ibrahim Hadi Wibisono
Dosen     : —
Tanggal   : 04 Juli 2026 (v1.0.1 — patch Fase 4)

Konsep yang diintegrasikan (Bab 2–13):
  Bab  2 → Variabel & Tipe Data     : semua variabel program
  Bab  3 → Seleksi (if/elif/else)   : routing menu, validasi, logika bisnis
  Bab  4 → Perulangan (for/while)   : loop menu, iterasi keranjang/riwayat
  Bab  5 → Fungsi                   : seluruh fungsi modular (Top-Down Design)
  Bab  6 → String                   : format struk, f-string, ljust/rjust
  Bab  7 → List/Tuple               : keranjang, riwayat, hasil pencarian
  Bab  8 → Dictionary/Set           : menu_produk, data transaksi, set kode unik
  Bab  9 → Searching                : linear search & binary search di menu produk
  Bab 10 → Sorting                  : sort produk by harga/nama/stok (bubble + built-in)
  Bab 11 → Rekursi                  : hitung_total_rekursif(), tampilkan_laporan_rekursif()
  Bab 12 → Big-O                    : analisis kompleksitas semua fungsi (lihat komentar)
  Bab 13 → AI Partner (CRIDE)       : lihat AI_USAGE_LOG di bawah & dokumen Fase 3
"""

# ─────────────────────────────────────────────
# BAB 2: VARIABEL & TIPE DATA
# ─────────────────────────────────────────────
# Konstanta program (str, int)
NAMA_WARUNG: str = "Warung Al Azhar"
LEBAR_STRUK: int = 44
VERSI: str = "1.0.1"

# Tipe data yang digunakan:
#   str   → nama produk, kode, ID transaksi
#   int   → harga, stok, qty, total, kembalian
#   float → (tidak digunakan — harga selalu bulat rupiah)
#   bool  → flag sukses/gagal di fungsi bisnis
#   list  → keranjang, riwayat_transaksi, hasil_sort
#   dict  → menu_produk (nested dict), item keranjang, record transaksi
#   tuple → return value ganda (sukses, kembalian)
#   set   → kumpulan kode produk unik (dipakai di cek duplikat)

from datetime import datetime  # stdlib saja, tanpa pip

## LAYER 3 — UTILITY & I/O FUNCTIONS

In [ ]:
def format_rupiah(angka: int) -> str:
    """
    Bab 6 — String: konversi integer ke format Rupiah Indonesia.
    Contoh: 10000 → 'Rp 10.000'
    Big-O: O(1) — hanya operasi string sederhana.
    """
    # f-string + replace: teknik string formatting (Bab 6)
    return f"Rp {angka:,}".replace(",", ".")


def garis(karakter: str = "=", lebar: int = LEBAR_STRUK) -> str:
    """
    Bab 6 — String: buat garis separator dari satu karakter.
    Big-O: O(n) di mana n = lebar.
    """
    return karakter * lebar  # string repetition (Bab 6)


def cetak_judul(teks: str) -> None:
    """
    Bab 6 — String: cetak judul rata tengah di dalam bingkai.
    Bab 5 — Fungsi: fungsi void dengan side-effect (print).
    """
    print(garis())
    print(teks.center(LEBAR_STRUK))  # str.center() (Bab 6)
    print(garis())


def validasi_input_angka(prompt: str, min_val: int = 0,
                          max_val: int = None) -> int:
    """
    Bab 3 — Seleksi: cek range min/max.
    Bab 4 — Perulangan (while): loop sampai input valid.
    Bab 5 — Fungsi: parameter dengan default value.
    Big-O: O(k) di mana k = jumlah percobaan input user → praktis O(1).
    """
    while True:                                    # Bab 4: while loop
        try:
            nilai = int(input(prompt))             # konversi str → int (Bab 2)
            if nilai < min_val:                    # Bab 3: seleksi
                print(f"  ⚠  Nilai minimal adalah {min_val}.")
                continue
            if max_val is not None and nilai > max_val:
                print(f"  ⚠  Nilai maksimal adalah {max_val}.")
                continue
            return nilai
        except ValueError:
            print("  ⚠  Input harus berupa angka bulat!")


def validasi_input_teks(prompt: str, min_len: int = 1) -> str:
    """
    Bab 6 — String: validasi panjang & strip whitespace.
    Bab 4 — Perulangan: loop sampai input tidak kosong.
    """
    while True:
        teks = input(prompt).strip()               # str.strip() (Bab 6)
        if len(teks) >= min_len:                   # len() pada string (Bab 6)
            return teks
        print(f"  ⚠  Input minimal {min_len} karakter.")


def tampilkan_menu_produk(menu_produk: dict) -> None:
    """
    Bab 8 — Dictionary: iterasi .items() untuk tampilkan semua produk.
    Bab 6 — String: f-string dengan alignment ljust/rjust.
    Bab 4 — Perulangan: for loop atas dict items.
    Big-O: O(n) di mana n = jumlah produk.
    """
    print(garis("-"))
    # Header dengan alignment string (Bab 6)
    print(f"  {'KODE':<6} {'NAMA PRODUK':<20} {'HARGA':>10} {'STOK':>5}")
    print(garis("-"))
    if not menu_produk:                            # Bab 3: seleksi dict kosong
        print("  (Belum ada produk)")
        return
    # Bab 4: for loop, Bab 8: .items() pada dictionary
    for kode, produk in menu_produk.items():
        status = "" if produk["stok"] > 0 else " ⚠ HABIS"
        print(
            f"  {kode:<6} "
            f"{produk['nama']:<20} "        # Bab 6: f-string ljust
            f"{format_rupiah(produk['harga']):>10} "
            f"{produk['stok']:>5}"
            f"{status}"
        )
    print(garis("-"))


def tampilkan_keranjang(keranjang: list) -> None:
    """
    Bab 7 — List: iterasi list of dict.
    Bab 6 — String: formatting alignment.
    Big-O: O(n) di mana n = item di keranjang.
    """
    if not keranjang:                              # Bab 3: cek list kosong
        print("  (Keranjang kosong)")
        return
    print(f"\n  {'ITEM':<22} {'QTY':>4} {'SUBTOTAL':>12}")
    print(garis("-"))
    for item in keranjang:                         # Bab 4: for atas list (Bab 7)
        print(
            f"  {item['nama']:<22}"
            f"{item['qty']:>4}"
            f"{format_rupiah(item['subtotal']):>12}"
        )


def cetak_struk(keranjang: list, total: int, kembalian: int,
                uang_masuk: int, id_trx: str) -> None:
    """
    Bab 6 — String: seluruh teknik formatting string (center, ljust, rjust, f-string).
    Bab 7 — List: iterasi list keranjang untuk baris item.
    Bab 5 — Fungsi: prosedur dengan multiple parameter.
    Big-O: O(n) di mana n = jumlah item keranjang.
    """
    print("\n" + garis())
    print(NAMA_WARUNG.center(LEBAR_STRUK))          # str.center() (Bab 6)
    print("Jl. Contoh No.1, Jakarta Selatan".center(LEBAR_STRUK))
    print(garis())
    # f-string dengan multiple variabel (Bab 6)
    waktu_sekarang = datetime.now().strftime("%d-%m-%Y  %H:%M:%S")
    print(f"  No      : {id_trx}")
    print(f"  Tanggal : {waktu_sekarang}")
    print(garis("-"))
    print(f"  {'ITEM':<22}{'QTY':>4}{'SUBTOTAL':>14}")
    print(garis("-"))
    # Bab 4: for loop, Bab 7: iterasi list of dict
    for item in keranjang:
        nama_col = item["nama"][:22].ljust(22)      # str.ljust() (Bab 6)
        qty_col  = str(item["qty"]).rjust(4)        # str.rjust() (Bab 6)
        sub_col  = format_rupiah(item["subtotal"]).rjust(14)
        print(f"  {nama_col}{qty_col}{sub_col}")
    print(garis("-"))
    # str.ljust + rjust untuk alignment kolom kanan (Bab 6)
    print(f"  {'TOTAL':<26}{format_rupiah(total):>14}")
    print(f"  {'BAYAR':<26}{format_rupiah(uang_masuk):>14}")
    print(f"  {'KEMBALI':<26}{format_rupiah(kembalian):>14}")
    print(garis())
    print("Terima kasih sudah berbelanja!".center(LEBAR_STRUK))
    print("Barang yang dibeli tidak dapat".center(LEBAR_STRUK))
    print("dikembalikan.".center(LEBAR_STRUK))
    print(garis())


def simpan_riwayat(id_trx: str, keranjang: list, total: int,
                   riwayat: list) -> None:
    """
    Bab 7 — List: append() ke list riwayat.
    Bab 8 — Dictionary: membuat dict record transaksi.
    Big-O: O(n) untuk copy keranjang; O(1) untuk append.
    """
    record = {                                     # Bab 8: membuat dict baru
        "id"    : id_trx,
        "waktu" : datetime.now().strftime("%H:%M:%S"),
        "total" : total,
        "items" : list(keranjang)                  # Bab 7: shallow copy list
    }
    riwayat.append(record)                         # Bab 7: list.append() O(1)


def tampilkan_riwayat(riwayat: list) -> None:
    """
    Bab 7 — List: enumerate() untuk indeks + iterasi.
    Bab 6 — String: formatting output laporan.
    Big-O: O(n) di mana n = jumlah transaksi.
    """
    cetak_judul("LAPORAN TRANSAKSI — SESI INI")
    if not riwayat:
        print("  Belum ada transaksi.")
        return
    print(f"  {'No.':<4} {'ID':<10} {'Waktu':<10} {'Total':>14}")
    print(garis("-"))
    for i, trx in enumerate(riwayat, start=1):    # Bab 4: enumerate (Bab 7)
        print(
            f"  {i:<4} "
            f"{trx['id']:<10} "
            f"{trx['waktu']:<10} "
            f"{format_rupiah(trx['total']):>14}"
        )
    print(garis("-"))
    total_sesi = sum(t["total"] for t in riwayat)  # generator expr (Bab 7)
    print(f"  Jumlah Transaksi : {len(riwayat)}")
    print(f"  Total Pendapatan : {format_rupiah(total_sesi)}")
    print(garis())

## BAB 9: SEARCHING

In [ ]:
def cari_produk_linear(keyword: str, menu_produk: dict) -> list:
    """
    Bab 9 — Searching: LINEAR SEARCH pada dict values.
    Cocok untuk data kecil (<100 produk) tanpa pra-sortir.
    Big-O: O(n) — worst case periksa semua produk.

    Args:
        keyword: string pencarian (nama atau kode, case-insensitive)
        menu_produk: dict produk master
    Returns:
        list of tuple (kode, produk_dict) yang cocok
    """
    hasil = []                                     # Bab 7: list untuk hasil
    kw = keyword.lower().strip()                   # Bab 6: str.lower(), strip()
    if not kw:                                     # Bab 3: guard keyword kosong (FIX)
        return hasil                               # kembalikan list kosong, bukan semua produk
    # Bab 9: Linear search — periksa satu per satu (Bab 4: for loop)
    for kode, produk in menu_produk.items():       # Bab 8: .items()
        nama_lower = produk["nama"].lower()        # Bab 6: case-insensitive
        if kw in nama_lower or kw == kode.lower(): # Bab 3: kondisi OR
            hasil.append((kode, produk))           # Bab 7: append tuple
    return hasil


def cari_produk_binary(kode_target: str, daftar_terurut: list) -> int:
    """
    Bab 9 — Searching: BINARY SEARCH pada list produk terurut by kode.
    Prasyarat: daftar_terurut sudah diurutkan by kode (ascending).
    Big-O: O(log n) — jauh lebih cepat dari linear untuk data besar.

    Args:
        kode_target: kode produk yang dicari (str)
        daftar_terurut: list of tuple (kode, produk) sudah terurut by kode
    Returns:
        indeks jika ditemukan, -1 jika tidak
    """
    kiri, kanan = 0, len(daftar_terurut) - 1      # Bab 2: variabel int
    while kiri <= kanan:                            # Bab 4: while loop
        tengah = (kiri + kanan) // 2               # Bab 2: integer division
        kode_tengah = daftar_terurut[tengah][0]    # Bab 7: indexing tuple
        if kode_tengah == kode_target:             # Bab 3: seleksi
            return tengah                          # found
        elif kode_tengah < kode_target:            # Bab 3: elif
            kiri = tengah + 1                      # cari di kanan
        else:
            kanan = tengah - 1                     # cari di kiri
    return -1                                      # not found

## BAB 10: SORTING

In [ ]:
def bubble_sort_produk(daftar: list, kunci: str = "harga",
                       ascending: bool = True) -> list:
    """
    Bab 10 — Sorting: implementasi BUBBLE SORT manual.
    Digunakan untuk menunjukkan konsep dasar sorting (tidak dipakai di menu prod).
    Big-O: O(n²) — kuadratik, tidak efisien untuk data besar.

    Args:
        daftar: list of tuple (kode, produk_dict)
        kunci: field untuk sort ('harga', 'nama', 'stok')
        ascending: True = kecil ke besar
    Returns:
        list terurut (copy baru)
    """
    hasil = list(daftar)                           # Bab 7: copy list
    n = len(hasil)
    # Bab 4: nested for loop (ciri khas bubble sort)
    for i in range(n - 1):                        # pass ke-i
        for j in range(n - 1 - i):                # bandingkan pasangan
            val_j   = hasil[j][1][kunci]
            val_j1  = hasil[j + 1][1][kunci]
            # Bab 3: kondisi swap
            kondisi_swap = val_j > val_j1 if ascending else val_j < val_j1
            if kondisi_swap:
                hasil[j], hasil[j + 1] = hasil[j + 1], hasil[j]  # swap (Bab 7)
    return hasil


def sort_produk_builtin(menu_produk: dict, kunci: str = "harga",
                        ascending: bool = True) -> list:
    """
    Bab 10 — Sorting: TIMSORT via sorted() built-in Python.
    Big-O: O(n log n) — jauh lebih efisien dari bubble sort.
    Dipakai untuk tampilan menu produk terurut.

    Args:
        menu_produk: dict produk master
        kunci: 'harga', 'nama', atau 'stok' — kunci lain akan fallback ke 'nama'
        ascending: urutan ascending/descending
    Returns:
        list of tuple (kode, produk_dict) terurut

    FIX v1.0.1: tambah validasi kunci — fallback ke 'nama' jika kunci tidak valid.
    """
    KUNCI_VALID = {"harga", "nama", "stok"}        # Bab 8: set untuk O(1) lookup
    if kunci not in KUNCI_VALID:                   # Bab 3: guard kunci invalid (FIX)
        kunci = "nama"                             # fallback default
    daftar = list(menu_produk.items())             # Bab 7: dict → list of tuples
    # Bab 5: lambda sebagai key function (Bab 10: sorting key)
    return sorted(daftar, key=lambda x: x[1][kunci], reverse=not ascending)

## BAB 11: REKURSI

In [ ]:
def hitung_total_rekursif(keranjang: list, indeks: int = 0) -> int:
    """
    Bab 11 — Rekursi: hitung total dengan pendekatan rekursif.
    Menunjukkan konsep base case & recursive case.
    Big-O: O(n) — n panggilan rekursif sesuai panjang keranjang.

    Base case  : indeks >= len(keranjang) → return 0
    Rekursif   : subtotal[indeks] + hitung_total_rekursif(indeks+1)
    """
    # Bab 3: Seleksi — BASE CASE (Bab 11)
    if indeks >= len(keranjang):
        return 0
    # Bab 11: RECURSIVE CASE — kurangi masalah jadi lebih kecil
    return keranjang[indeks]["subtotal"] + hitung_total_rekursif(keranjang, indeks + 1)


def tampilkan_laporan_rekursif(riwayat: list, indeks: int = 0,
                                total_akumulasi: int = 0) -> int:
    """
    Bab 11 — Rekursi: tampilkan laporan & hitung total secara rekursif.
    Big-O: O(n) — n = jumlah transaksi dalam riwayat.

    Base case : indeks >= len(riwayat) → print total & return
    Rekursif  : print baris ke-indeks, lalu panggil indeks+1
    """
    if indeks >= len(riwayat):                     # Bab 11: BASE CASE
        return total_akumulasi
    trx = riwayat[indeks]                          # Bab 7: indexing list
    print(
        f"  {indeks+1:<4} "
        f"{trx['id']:<10} "
        f"{trx['waktu']:<10} "
        f"{format_rupiah(trx['total']):>14}"
    )
    # Bab 11: RECURSIVE CALL dengan akumulasi total
    return tampilkan_laporan_rekursif(
        riwayat, indeks + 1, total_akumulasi + trx["total"]
    )

## LAYER 2 — CORE BUSINESS LOGIC

In [ ]:
def tambah_item_ke_keranjang(kode: str, qty: int,
                              menu_produk: dict, keranjang: list) -> tuple:
    """
    Bab 8 — Dictionary: lookup O(1) by kode.
    Bab 7 — List: linear search di keranjang + append.
    Bab 9 — Searching: linear search sebelum append (hindari duplikat).
    Bab 3 — Seleksi: validasi kode & stok.
    Big-O: O(n) — n = item saat ini di keranjang (karena linear search).

    Returns:
        tuple (sukses: bool, pesan: str)   ← Bab 2: tuple (Bab 7)
    """
    # Bab 8: O(1) hash lookup di dictionary
    if kode not in menu_produk:                    # Bab 3: seleksi
        return (False, f"Kode '{kode}' tidak ditemukan.")   # Bab 7: tuple

    produk = menu_produk[kode]                     # Bab 8: dict access

    # Bab 3: validasi stok
    if qty <= 0:
        return (False, "Kuantitas harus lebih dari 0.")
    if produk["stok"] < qty:
        return (False, f"Stok tidak cukup. Tersisa: {produk['stok']} unit.")

    # Bab 9: Linear search — cek apakah kode sudah di keranjang
    for item in keranjang:                         # Bab 4: for (Bab 7: list)
        if item["kode"] == kode:                   # Bab 3: seleksi
            # Update qty dan subtotal jika sudah ada
            if produk["stok"] < item["qty"] + qty:
                return (False, f"Total qty melebihi stok ({produk['stok']} unit).")
            item["qty"] += qty
            item["subtotal"] = item["qty"] * produk["harga"]
            return (True, f"'{produk['nama']}' diperbarui (total: {item['qty']} unit).")

    # Item baru → append ke keranjang
    keranjang.append({                             # Bab 7: list.append()
        "kode"    : kode,                          # Bab 8: dict key-value
        "nama"    : produk["nama"],
        "harga"   : produk["harga"],
        "qty"     : qty,
        "subtotal": produk["harga"] * qty
    })
    return (True, f"'{produk['nama']}' x{qty} ditambahkan ke keranjang.")


def hapus_item_keranjang(kode: str, keranjang: list) -> tuple:
    """
    Bab 7 — List: enumerate + pop() untuk hapus by indeks.
    Bab 9 — Searching: linear search untuk temukan indeks.
    Big-O: O(n) — linear search + pop O(n) worst case.
    """
    # Bab 9: linear search dengan enumerate (Bab 7)
    for i, item in enumerate(keranjang):
        if item["kode"] == kode:                   # Bab 3: seleksi
            nama = item["nama"]
            keranjang.pop(i)                       # Bab 7: list.pop(i)
            return (True, f"'{nama}' dihapus dari keranjang.")
    return (False, f"Kode '{kode}' tidak ada di keranjang.")


def hitung_total(keranjang: list) -> int:
    """
    Bab 7 — List: iterasi + akumulasi.
    Bab 4 — Perulangan: for loop.
    Big-O: O(n) — n = item di keranjang.
    (Versi iteratif — versi rekursif ada di hitung_total_rekursif)
    """
    total = 0
    for item in keranjang:                         # Bab 4: for loop (Bab 7)
        total += item["subtotal"]                  # akumulasi
    return total


def proses_pembayaran(total: int, uang_masuk: int) -> tuple:
    """
    Bab 2 — Tipe Data: aritmetika integer.
    Bab 3 — Seleksi: validasi uang >= total.
    Big-O: O(1) — operasi matematika konstan.

    Returns:
        tuple (sukses: bool, kembalian: int, pesan: str)
    """
    if uang_masuk < total:                         # Bab 3: seleksi
        kurang = total - uang_masuk
        return (False, 0, f"Uang kurang {format_rupiah(kurang)}.")
    kembalian = uang_masuk - total                 # Bab 2: aritmetika int
    return (True, kembalian, "Pembayaran berhasil.")


def update_stok(keranjang: list, menu_produk: dict) -> None:
    """
    Bab 8 — Dictionary: modifikasi nilai dict in-place.
    Bab 7 — List: iterasi list keranjang.
    Bab 3 — Seleksi: guard kode tidak ditemukan (defensive fix).
    Big-O: O(n) — n = item keranjang; setiap lookup dict O(1).

    FIX v1.0.1: tambah guard kode not in menu_produk untuk mencegah KeyError
    jika keranjang berisi kode yang tidak ada di menu (data corrupt / test case).
    """
    for item in keranjang:                         # Bab 4: for (Bab 7)
        kode = item["kode"]
        if kode not in menu_produk:                # Bab 3: defensive guard (FIX)
            continue                               # skip item yang kodanya tidak valid
        menu_produk[kode]["stok"] -= item["qty"]   # Bab 8: dict mutation


def get_kode_unik(menu_produk: dict) -> set:
    """
    Bab 8 — Set: kumpulkan semua kode produk ke dalam set.
    Digunakan untuk validasi kode duplikat saat tambah produk baru.
    Big-O: O(n) untuk build set; O(1) untuk membership check.
    """
    return set(menu_produk.keys())                 # Bab 8: dict.keys() → set

## BAB 12: BIG-O — ANALISIS KOMPLEKSITAS

In [ ]:
# Tabel ringkasan Big-O seluruh fungsi utama:
#
#  Fungsi                       | Waktu  | Ruang  | Keterangan
#  ─────────────────────────────|────────|────────|──────────────────────────
#  format_rupiah()              | O(1)   | O(1)   | Konversi string konstant
#  validasi_input_angka()       | O(k)   | O(1)   | k = percobaan user
#  tambah_item_ke_keranjang()   | O(n)   | O(1)   | n = item keranjang
#  hapus_item_keranjang()       | O(n)   | O(1)   | linear search + pop
#  hitung_total()               | O(n)   | O(1)   | satu pass keranjang
#  hitung_total_rekursif()      | O(n)   | O(n)   | O(n) call stack
#  proses_pembayaran()          | O(1)   | O(1)   | aritmetika int
#  update_stok()                | O(n)   | O(1)   | n iter + O(1) dict lookup
#  cari_produk_linear()         | O(n)   | O(k)   | n produk, k hasil
#  cari_produk_binary()         | O(log n)| O(1)  | prasyarat: sudah terurut
#  bubble_sort_produk()         | O(n²)  | O(n)   | nested loop
#  sort_produk_builtin()        | O(n log n)| O(n) | Timsort Python
#  simpan_riwayat()             | O(n)   | O(n)   | copy keranjang
#  tampilkan_riwayat()          | O(n)   | O(1)   | satu pass riwayat
#  tampilkan_laporan_rekursif() | O(n)   | O(n)   | call stack depth n


def tampilkan_analisis_bigO() -> None:
    """
    Bab 12 — Big-O: tampilkan tabel analisis kompleksitas ke layar.
    Bab 5 — Fungsi, Bab 6 — String, Bab 7 — List of tuple.
    """
    tabel = [                                       # Bab 7: list of tuple
        ("format_rupiah",              "O(1)",      "O(1)"),
        ("validasi_input_angka",       "O(k)",      "O(1)"),
        ("tambah_item_ke_keranjang",   "O(n)",      "O(1)"),
        ("hapus_item_keranjang",       "O(n)",      "O(1)"),
        ("hitung_total (iteratif)",    "O(n)",      "O(1)"),
        ("hitung_total (rekursif)",    "O(n)",      "O(n)"),
        ("proses_pembayaran",          "O(1)",      "O(1)"),
        ("update_stok",                "O(n)",      "O(1)"),
        ("cari_produk_linear",         "O(n)",      "O(k)"),
        ("cari_produk_binary",         "O(log n)",  "O(1)"),
        ("bubble_sort_produk",         "O(n²)",     "O(n)"),
        ("sort_produk_builtin",        "O(n log n)","O(n)"),
    ]
    cetak_judul("ANALISIS KOMPLEKSITAS BIG-O")
    print(f"  {'FUNGSI':<35} {'WAKTU':>8}  {'RUANG':>8}")
    print(garis("-"))
    for nama, waktu, ruang in tabel:               # Bab 4: for, Bab 7: unpack tuple
        print(f"  {nama:<35} {waktu:>8}  {ruang:>8}")
    print(garis())
    input("\n  [ENTER] Kembali ke menu...")

## LAYER 1 — CONTROLLER FUNCTIONS

In [ ]:
def menu_stok(menu_produk: dict) -> None:
    """
    Bab 3 — Seleksi: routing pilihan menu stok.
    Bab 4 — Perulangan: while loop.
    Bab 8 — Dictionary: tambah & update entry.
    Bab 8 — Set: cek duplikat kode produk.
    """
    while True:
        cetak_judul("MANAJEMEN STOK PRODUK")
        tampilkan_menu_produk(menu_produk)
        print("\n  [1] Tambah Produk Baru")
        print("  [2] Update Harga Produk")
        print("  [3] Update Jumlah Stok")
        print("  [4] Tampilkan Produk Terurut (Sort Demo)")
        print("  [5] Cari Produk (Search Demo)")
        print("  [6] Kembali ke Menu Utama")
        pilihan = validasi_input_angka("\n  Pilihan [1-6]: ", 1, 6)

        if pilihan == 1:
            # Tambah produk baru
            kode_set = get_kode_unik(menu_produk)  # Bab 8: set
            kode = validasi_input_teks("  Kode produk baru (maks 6 char): ", 1).upper()[:6]
            if kode in kode_set:                   # Bab 8: O(1) set membership
                print(f"  ⚠  Kode '{kode}' sudah ada.")
                input("  [ENTER] lanjut..."); continue
            nama   = validasi_input_teks("  Nama produk: ")
            harga  = validasi_input_angka("  Harga (Rp): ", min_val=1)
            stok   = validasi_input_angka("  Stok awal : ", min_val=0)
            menu_produk[kode] = {                  # Bab 8: dict assignment
                "nama" : nama,
                "harga": harga,
                "stok" : stok
            }
            print(f"  ✓  Produk '{nama}' berhasil ditambahkan.")

        elif pilihan == 2:
            # Update harga
            kode = validasi_input_teks("  Kode produk: ").upper()
            if kode not in menu_produk:
                print("  ⚠  Kode tidak ditemukan.")
            else:
                harga_baru = validasi_input_angka("  Harga baru (Rp): ", min_val=1)
                menu_produk[kode]["harga"] = harga_baru  # Bab 8: dict update
                print(f"  ✓  Harga '{menu_produk[kode]['nama']}' diperbarui.")

        elif pilihan == 3:
            # Update stok
            kode = validasi_input_teks("  Kode produk: ").upper()
            if kode not in menu_produk:
                print("  ⚠  Kode tidak ditemukan.")
            else:
                stok_baru = validasi_input_angka("  Stok baru : ", min_val=0)
                menu_produk[kode]["stok"] = stok_baru  # Bab 8
                print(f"  ✓  Stok '{menu_produk[kode]['nama']}' diperbarui.")

        elif pilihan == 4:
            # Demo SORTING (Bab 10)
            cetak_judul("DEMO SORTING PRODUK")
            print("  Urutkan berdasarkan:")
            print("  [1] Harga (termurah)   [2] Harga (termahal)")
            print("  [3] Nama (A-Z)         [4] Stok (terbanyak)")
            p = validasi_input_angka("  Pilih [1-4]: ", 1, 4)
            kunci_map = {1: ("harga", True), 2: ("harga", False),
                         3: ("nama", True),  4: ("stok", False)}
            kunci, asc = kunci_map[p]

            # Tampilkan bubble sort vs built-in sort
            print(f"\n  — Bubble Sort (O(n²)) —")
            daftar_items = list(menu_produk.items())
            hasil_bubble = bubble_sort_produk(daftar_items, kunci, asc)
            for kode, prod in hasil_bubble:        # Bab 4: for, Bab 7: unpack
                print(f"  {kode:<6} {prod['nama']:<20} "
                      f"{format_rupiah(prod['harga']):>10} stok:{prod['stok']:>4}")

            print(f"\n  — Built-in Timsort (O(n log n)) —")
            hasil_builtin = sort_produk_builtin(menu_produk, kunci, asc)
            for kode, prod in hasil_builtin:
                print(f"  {kode:<6} {prod['nama']:<20} "
                      f"{format_rupiah(prod['harga']):>10} stok:{prod['stok']:>4}")

        elif pilihan == 5:
            # Demo SEARCHING (Bab 9)
            keyword = validasi_input_teks("  Kata kunci pencarian: ")
            hasil_linear = cari_produk_linear(keyword, menu_produk)  # O(n)
            print(f"\n  — Linear Search (O(n)) — ditemukan {len(hasil_linear)} produk:")
            if hasil_linear:
                for kode, prod in hasil_linear:    # Bab 7: unpack tuple
                    print(f"  {kode} | {prod['nama']} | {format_rupiah(prod['harga'])}")
            else:
                print("  Tidak ditemukan.")

            # Binary search (hanya untuk kode persis)
            print(f"\n  — Binary Search (O(log n)) — cari kode persis:")
            kode_cari  = keyword.upper()
            terurut    = sort_produk_builtin(menu_produk, "nama")      # presort by nama
            terurut_kode = sorted(menu_produk.items(), key=lambda x: x[0])  # by kode
            idx = cari_produk_binary(kode_cari, terurut_kode)
            if idx != -1:
                k, p = terurut_kode[idx]
                print(f"  Ditemukan di indeks {idx}: {k} | {p['nama']}")
            else:
                print(f"  Kode '{kode_cari}' tidak ditemukan (binary search).")

        else:  # pilihan == 6
            break

        input("\n  [ENTER] lanjut...")


def menu_kasir(menu_produk: dict, riwayat: list,
               counter_id: list) -> None:
    """
    Bab 3 — Seleksi: routing aksi kasir.
    Bab 4 — Perulangan: while True loop transaksi.
    Bab 7 — List: keranjang sebagai list of dict.
    counter_id: list[int] — mutable container agar bisa dimodifikasi dari luar.
    """
    keranjang: list = []                           # Bab 7: list kosong

    while True:
        id_trx = f"TRX-{counter_id[0]:03d}"       # Bab 6: f-string zero-pad
        cetak_judul(f"TRANSAKSI BARU — #{id_trx}")
        tampilkan_menu_produk(menu_produk)

        print("\n  ═══ KERANJANG BELANJA ═══")
        tampilkan_keranjang(keranjang)
        if keranjang:
            total_sementara = hitung_total(keranjang)
            print(f"\n  SUBTOTAL : {format_rupiah(total_sementara)}")

        print("\n  [1] Tambah Item    [3] Hapus Item")
        print("  [2] Proses Bayar   [4] Demo: Total Rekursif")
        print("  [5] Batal Transaksi")
        pilihan = validasi_input_angka("\n  Pilihan [1-5]: ", 1, 5)

        if pilihan == 1:
            kode = validasi_input_teks("  Kode produk: ").upper()
            qty  = validasi_input_angka("  Jumlah     : ", min_val=1)
            sukses, pesan = tambah_item_ke_keranjang(kode, qty, menu_produk, keranjang)
            print(f"  {'✓' if sukses else '⚠'}  {pesan}")

        elif pilihan == 2:
            if not keranjang:
                print("  ⚠  Keranjang masih kosong!")
                input("  [ENTER] lanjut..."); continue

            total = hitung_total(keranjang)
            print(f"\n  Total belanja : {format_rupiah(total)}")
            uang  = validasi_input_angka("  Uang dibayar : Rp ", min_val=0)
            sukses, kembalian, pesan = proses_pembayaran(total, uang)

            if sukses:
                update_stok(keranjang, menu_produk)  # Bab 8: update dict
                cetak_struk(keranjang, total, kembalian, uang, id_trx)
                simpan_riwayat(id_trx, keranjang, total, riwayat)
                counter_id[0] += 1                 # Bab 2: increment counter
                keranjang.clear()                  # Bab 7: list.clear()
                input("  [ENTER] Lanjut transaksi baru...")
                break
            else:
                print(f"  ⚠  {pesan}")

        elif pilihan == 3:
            kode = validasi_input_teks("  Kode produk yang dihapus: ").upper()
            sukses, pesan = hapus_item_keranjang(kode, keranjang)
            print(f"  {'✓' if sukses else '⚠'}  {pesan}")

        elif pilihan == 4:
            # Demo REKURSI (Bab 11)
            if keranjang:
                total_iter  = hitung_total(keranjang)
                total_rekur = hitung_total_rekursif(keranjang)
                print(f"\n  Total (iteratif) : {format_rupiah(total_iter)}")
                print(f"  Total (rekursif) : {format_rupiah(total_rekur)}")
                print("  Kedua metode menghasilkan nilai yang sama ✓")
            else:
                print("  ⚠  Keranjang kosong — tambahkan item terlebih dahulu.")

        else:  # pilihan == 5
            if keranjang:
                konfirm = input("  Batalkan transaksi? Data hilang. [y/N]: ")
                if konfirm.lower() == "y":
                    keranjang.clear()
                    print("  Transaksi dibatalkan.")
                    break
            else:
                break

        input("\n  [ENTER] lanjut...")


def menu_laporan(riwayat: list) -> None:
    """
    Bab 7 — List: iterasi riwayat.
    Bab 11 — Rekursi: tampilkan_laporan_rekursif sebagai opsi demo.
    """
    while True:
        cetak_judul("LAPORAN TRANSAKSI — SESI INI")
        print("  [1] Laporan Iteratif  (Bab 4 — for loop)")
        print("  [2] Laporan Rekursif  (Bab 11 — rekursi)")
        print("  [3] Analisis Big-O    (Bab 12)")
        print("  [4] Kembali ke Menu Utama")
        pilihan = validasi_input_angka("\n  Pilihan [1-4]: ", 1, 4)

        if pilihan == 1:
            tampilkan_riwayat(riwayat)             # iteratif

        elif pilihan == 2:
            # Demo rekursi (Bab 11)
            cetak_judul("LAPORAN REKURSIF (Bab 11)")
            if not riwayat:
                print("  Belum ada transaksi.")
            else:
                print(f"  {'No.':<4} {'ID':<10} {'Waktu':<10} {'Total':>14}")
                print(garis("-"))
                total_rek = tampilkan_laporan_rekursif(riwayat)
                print(garis("-"))
                print(f"  Total Pendapatan (rekursif) : {format_rupiah(total_rek)}")

        elif pilihan == 3:
            tampilkan_analisis_bigO()
            continue

        else:
            break

        input("\n  [ENTER] lanjut...")


def menu_utama(menu_produk: dict, riwayat: list,
               counter_id: list) -> None:
    """
    Bab 3 — Seleksi: routing ke sub-menu.
    Bab 4 — Perulangan: while True loop utama program.
    Bab 5 — Fungsi: memanggil fungsi controller.
    """
    while True:                                    # Bab 4: while loop
        cetak_judul(f"{NAMA_WARUNG}  v{VERSI}")
        print("  Selamat datang, Operator!")
        print(garis("-"))
        print("  [1] Proses Transaksi (Kasir)")
        print("  [2] Manajemen Stok Produk")
        print("  [3] Laporan Transaksi")
        print("  [4] Keluar Program")
        print(garis("-"))
        pilihan = validasi_input_angka("  Pilihan Anda [1-4]: ", 1, 4)

        if pilihan == 1:
            menu_kasir(menu_produk, riwayat, counter_id)  # Bab 5: call fungsi
        elif pilihan == 2:
            menu_stok(menu_produk)
        elif pilihan == 3:
            menu_laporan(riwayat)
        else:                                      # pilihan == 4
            print("\n  Terima kasih. Sampai jumpa! 👋")
            break

## LAYER 0 — ENTRY POINT

In [ ]:
def init_menu_default() -> dict:
    """
    Bab 8 — Dictionary: inisialisasi dict of dict produk default.
    Bab 2 — Tipe Data: str, int di dalam dict.
    """
    return {                                       # Bab 8: nested dictionary
        "P001": {"nama": "Indomie Goreng",    "harga": 3500,  "stok": 50},
        "P002": {"nama": "Aqua 600ml",        "harga": 4000,  "stok": 30},
        "P003": {"nama": "Chitato 68g",       "harga": 8500,  "stok": 20},
        "P004": {"nama": "Teh Botol Sosro",   "harga": 5000,  "stok": 40},
        "P005": {"nama": "Kopi ABC Susu",     "harga": 4500,  "stok": 35},
        "P006": {"nama": "Roti Tawar Sari",   "harga": 12000, "stok": 15},
        "P007": {"nama": "Mie Sedaap Kari",   "harga": 3000,  "stok": 45},
        "P008": {"nama": "Sabun Lifebuoy",    "harga": 7500,  "stok": 25},
    }


def main() -> None:
    """
    Bab 5 — Fungsi: main() sebagai entry point (konvensi Python).
    Bab 2 — Tipe Data: inisialisasi variabel bertipe dict, list, int.
    Bab 3 — Seleksi: try-except untuk KeyboardInterrupt.

    AI Usage Log — Bab 13 (CRIDE Framework):
    - C (Context)   : Program kasir warung CLI dengan 12 konsep bab
    - R (Request)   : "Bantu implementasi docstring dan error handling"
    - I (Iterate)   : Direvisi 3x — tambahkan tipe hint, perbaiki pesan error
    - D (Document)  : Semua AI interaction dicatat di dokumen Fase 3
    - E (Evaluate)  : Logika inti (bisnis, rekursi, sort, search) ditulis sendiri
    """
    # Bab 2: inisialisasi variabel dengan tipe yang sesuai
    menu_produk: dict = init_menu_default()        # Bab 8: dict
    riwayat    : list = []                         # Bab 7: list
    counter_id : list = [1]                        # list sebagai mutable container

    print("\n" + garis("═"))
    print(f"  {NAMA_WARUNG} — Sistem Kasir v{VERSI}".center(LEBAR_STRUK))
    print(f"  Algoritma & Pemrograman — Al Azhar 2026".center(LEBAR_STRUK))
    print(garis("═") + "\n")

    # Bab 3: try-except untuk exit bersih
    try:
        menu_utama(menu_produk, riwayat, counter_id)
    except KeyboardInterrupt:
        print("\n\n  Program dihentikan (Ctrl+C). Sampai jumpa!")

## GUARD: hanya jalankan jika file dieksekusi langsung

In [ ]:
if __name__ == "__main__":
    main()